# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fatima-38/Flyrank_ML_Internship_Projects/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

We build a robust feature vector by handling missing values through median imputation for numerical metrics, mode encoding for categorical fields, and standard scaling. Categorical columns are processed via one-hot encoding to prevent ordinal bias.

In [1]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Sample dataframe initialization for pipeline execution
def build_feature_pipeline(df):
    numeric_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
    categorical_features = df.select_dtypes(include=['object', 'category']).columns.tolist()

    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ])

    return preprocessor

print("Feature vector transformation pipeline constructed successfully.")

Feature vector transformation pipeline constructed successfully.


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

- user_engagement_score: Represents the normalized frequency of platform interactions over a 30-day sliding window. Missing values are imputed using the median. Available at prediction time.

- account_tier: Denotes the subscription level (Free, Pro, Enterprise). Missing values are filled with the 'missing' category label and one-hot encoded. Available at prediction time.

- session_duration_avg: The average duration of recent user sessions. Missing values are imputed via median strategy. Available at prediction time.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

**Leakage Verification Test:**
We test our feature set to ensure no target-derived columns (such as post-conversion flags or future event windows) are inadvertently included in the feature vector.

In [2]:
# The Leakage Hunt: Checking for forbidden columns containing target data or future windows
def check_for_leakage(df, feature_columns, target_column):
    forbidden_keywords = ['target', 'label', 'post_', 'future_', 'converted', 'outcome']
    leaky_features = []

    for col in feature_columns:
        if any(keyword in col.lower() for keyword in forbidden_keywords):
            leaky_features.append(col)

    assert len(leaky_features) == 0, f"Data Leakage Detected! Forbidden columns found: {leaky_features}"
    print("Leakage Hunt Passed: No label-derived or future-window features present in the feature vector.")

# Example execution test (mocking feature list)
mock_features = ['user_engagement_score', 'account_tier', 'session_duration_avg']
check_for_leakage(pd.DataFrame(columns=mock_features), mock_features, 'target')

Leakage Hunt Passed: No label-derived or future-window features present in the feature vector.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

- conversion_timestamp: Excluded to prevent direct target leakage since it occurs strictly after the prediction point.

- support_ticket_resolution_status: Excluded because it contains future outcome information not known at the exact moment of inference.

- device_ip_address: Excluded due to privacy constraints and high cardinality noise that degrades model generalization.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.